# EDA — CSI por receptor (rx0, rx1, rx2) y por actividad

Explora el dataset con **pandas** para ver qué captura cada ESP32 (rx0/rx1/rx2) en cada
actividad y detectar diferencias/problemas antes de entrenar.

In [ ]:
import os, glob
import numpy as np, pandas as pd, scipy.io as scio
import matplotlib.pyplot as plt

HERE = os.path.abspath('')
DATA = os.environ.get('CSI2PC_DATA', os.path.join(HERE, '..', 'mmfi_dataset'))
mats = sorted(glob.glob(os.path.join(DATA, '**', 'wifi-csi', '*.mat'), recursive=True))
print('frames:', len(mats))

# Cada .mat: CSIamp[3 rx, 114 subportadoras, 10 ventana]. Promediamos la ventana temporal.
filas = []
amp_cube = {}   # (accion, rx) -> lista de [114] por frame
for m in mats:
    acc = os.path.basename(os.path.dirname(os.path.dirname(m)))
    d = scio.loadmat(m)['CSIamp'].astype(np.float64)   # (3,114,10)
    d[np.isinf(d)] = np.nan
    amp = np.nanmean(d, axis=2)                         # (3,114) media en el tiempo
    for rx in range(3):
        filas.append({'accion': acc, 'rx': f'rx{rx}',
                      'amp_media': np.nanmean(amp[rx]), 'amp_std': np.nanstd(amp[rx])})
        amp_cube.setdefault((acc, rx), []).append(amp[rx])

df = pd.DataFrame(filas)
acciones = sorted(df['accion'].unique())
print('acciones:', acciones)
df.head()

## 1. Resumen por actividad y receptor (amplitud media y variabilidad)

In [ ]:
tabla = df.groupby(['accion', 'rx']).agg(
    amp_media=('amp_media', 'mean'),
    amp_std=('amp_std', 'mean'),
    n=('amp_media', 'size')).round(3)
display(tabla)
# variabilidad temporal: std de la amplitud ENTRE frames (mas alto = mas movimiento)
var_t = df.groupby(['accion','rx'])['amp_media'].std().unstack().round(3)
print('\nVariabilidad entre frames (std de amp_media; mas alto = mas movimiento):')
display(var_t)

## 2. Amplitud media por subportadora — comparar rx0/rx1/rx2 en cada actividad

In [ ]:
n = len(acciones)
fig, axes = plt.subplots(1, n, figsize=(4*n, 3.5), sharey=True)
if n == 1: axes = [axes]
for ax, acc in zip(axes, acciones):
    for rx in range(3):
        perfil = np.nanmean(np.array(amp_cube[(acc, rx)]), axis=0)  # media por subportadora
        ax.plot(perfil, label=f'rx{rx}')
    ax.set_title(acc); ax.set_xlabel('subportadora'); ax.grid(alpha=.3)
axes[0].set_ylabel('amplitud media'); axes[0].legend()
plt.tight_layout(); plt.show()

## 3. Mapa de calor: subportadora × frame (una actividad, los 3 rx)
Muestra cómo cambia el CSI en el tiempo. En actividades con movimiento se ven franjas/variación; en estático es casi plano.

In [ ]:
ACC = acciones[-1]   # cambia por la actividad que quieras inspeccionar
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for rx in range(3):
    M = np.array(amp_cube[(ACC, rx)])        # (n_frames, 114)
    im = axes[rx].imshow(M.T, aspect='auto', origin='lower', cmap='viridis')
    axes[rx].set_title(f'{ACC} - rx{rx}'); axes[rx].set_xlabel('frame'); axes[rx].set_ylabel('subportadora')
    plt.colorbar(im, ax=axes[rx], fraction=0.046)
plt.tight_layout(); plt.show()

## 4. Variabilidad temporal por subportadora (detecta movimiento y subportadoras muertas)

In [ ]:
fig, axes = plt.subplots(1, len(acciones), figsize=(4*len(acciones), 3.5), sharey=True)
if len(acciones) == 1: axes = [axes]
for ax, acc in zip(axes, acciones):
    for rx in range(3):
        M = np.array(amp_cube[(acc, rx)])
        ax.plot(M.std(axis=0), label=f'rx{rx}')   # std por subportadora entre frames
    ax.set_title(acc); ax.set_xlabel('subportadora'); ax.grid(alpha=.3)
axes[0].set_ylabel('std temporal'); axes[0].legend()
plt.suptitle('std por subportadora (alto = movimiento; 0 plano = subportadora muerta)')
plt.tight_layout(); plt.show()
# alerta de subportadoras muertas (std ~0 en todas las actividades)
for rx in range(3):
    muertas = [s for s in range(114) if all(np.array(amp_cube[(a,rx)]).std(axis=0)[s] < 1e-6 for a in acciones)]
    print(f'rx{rx}: subportadoras muertas = {len(muertas)}', muertas[:10])